In [6]:
import pandas as pd
import numpy as np
import os

# ============================================================
# CONFIGURACIÓN
# ============================================================
# Carpeta donde están los datos originales del INEI (módulos ENAHO)
datos_originales = r"C:\Users\Dafne\Documents\GitHub\Base-de-datos-investigacion-Economica\ENAHO"

# Carpeta donde se guardarán los resultados
base_resultados = r"C:\Users\Dafne\Documents\GitHub\Base-de-datos-investigacion-Economica\Base24_25"

# Crear carpeta de resultados si no existe
os.makedirs(base_resultados, exist_ok=True)

# ============================================================
# FUNCIONES AUXILIARES
# ============================================================

def encontrar_archivo(carpeta, inicio_nombre):
    for f in os.listdir(carpeta):
        if f.upper().replace(".CSV", "").startswith(inicio_nombre.upper()):
            return os.path.join(carpeta, f)
    raise FileNotFoundError(f"No se encontró {inicio_nombre} en {carpeta}")

def leer_csv_inei(ruta):
    for sep in [",", ";"]:
        for enc in ["utf-8-sig", "latin1"]:
            try:
                df = pd.read_csv(ruta, sep=sep, encoding=enc, low_memory=False, dtype=str)
                if len(df.columns) > 1:
                    df.columns = df.columns.str.strip()
                    return df
            except Exception:
                pass
    raise ValueError(f"No se pudo leer el archivo: {ruta}")

def to_num(serie):
    return pd.to_numeric(serie, errors="coerce")

def construir_llave_persona(df, llaves=None, nombre="llave_persona"):
    if llaves is None:
        llaves = ["CONGLOME", "VIVIENDA", "HOGAR", "CODPERSO"]
    df = df.copy()
    for c in llaves:
        df[f"_{c}_k"] = pd.to_numeric(df[c], errors="coerce").astype("Int64").astype(str)
    df[nombre] = df[[f"_{c}_k" for c in llaves]].agg("-".join, axis=1)
    return df.drop(columns=[f"_{c}_k" for c in llaves])

# Columnas de uso de billetera (P558H1_7 a P558H12_7)
COLS_USO_BILLETERA = [f"P558H{i}_7" for i in range(1, 13)]

def reparar_variables(df):
    df = df.copy()

    df["Edad"] = to_num(df["P208A"]) if "P208A" in df.columns else df.get("Edad")
    df["P507_num"] = to_num(df["P507"])
    df["P510A1_num"] = to_num(df["P510A1"])
    df["P511A_num"] = to_num(df["P511A"])

    # --- Ocupado ---
    df["Ocupado"] = df["P507_num"].notna().astype(int)

    # --- Informalidad ---
    tfnr = df["P507_num"] == 5
    trab_hogar = df["P507_num"] == 6
    sin_sunat = (df["P507_num"] == 2) & (df["P510A1_num"] == 3)
    dependiente = df["P507_num"].isin([3, 4])
    sin_contrato = df["P511A_num"] == 7

    df["Informal"] = (
        sin_sunat.fillna(False)
        | tfnr.fillna(False)
        | trab_hogar.fillna(False)
        | (dependiente & sin_contrato.fillna(False))
    ).astype(int)
    df.loc[df["Ocupado"] == 0, "Informal"] = pd.NA

    # --- Crédito formal ---
    df["P558E1_4"] = to_num(df.get("P558E1_4")).fillna(0)
    df["P558E1_9"] = to_num(df.get("P558E1_9")).fillna(0)
    df["CreditoFormal"] = ((df["P558E1_4"] == 4) | (df["P558E1_9"] == 9)).astype(int)

    # --- Tenencia de billetera ---
    df["P558E1_8"] = to_num(df.get("P558E1_8")).fillna(0)
    df["TenenciaBilletera"] = (df["P558E1_8"] == 8).astype(int)

    # --- Uso de billetera ---
    cols_uso_presentes = [c for c in COLS_USO_BILLETERA if c in df.columns]
    if cols_uso_presentes:
        uso_flags = pd.concat(
            [to_num(df[c]).fillna(0) == 7 for c in cols_uso_presentes], axis=1
        )
        df["UsoBilletera"] = uso_flags.any(axis=1).astype(int)
    else:
        df["UsoBilletera"] = pd.NA

    # --- Crédito informal ---
    if "P558G7" in df.columns:
        df["CreditoInformal"] = (to_num(df["P558G7"]).fillna(0) == 7).astype(int)

    return df

def construir_master(anio, carpetas):
    print(f"Cargando módulos {anio}...")

    ruta_200 = encontrar_archivo(*carpetas["Mod200"])
    df200 = leer_csv_inei(ruta_200)[["CONGLOME", "VIVIENDA", "HOGAR", "CODPERSO", "P203", "P207", "P208A"]]

    ruta_300 = encontrar_archivo(*carpetas["Mod300"])
    df300 = leer_csv_inei(ruta_300)[["CONGLOME", "VIVIENDA", "HOGAR", "CODPERSO", "P301A", "P301B"]]

    ruta_500 = encontrar_archivo(*carpetas["Mod500"])
    df500_full = leer_csv_inei(ruta_500)
    cols_500 = ["CONGLOME", "VIVIENDA", "HOGAR", "CODPERSO",
                "P507", "P510A1", "P511A", "P512A",
                "P558E1_4", "P558E1_8", "P558E1_9", "P558G7"] + COLS_USO_BILLETERA
    df500 = df500_full[[c for c in cols_500 if c in df500_full.columns]]

    ruta_100 = encontrar_archivo(*carpetas["Mod100"])
    df100 = leer_csv_inei(ruta_100)
    cols_100 = ["CONGLOME", "VIVIENDA", "HOGAR", "DOMINIO", "ESTRATO", "PANEL"]
    for fac in ["FACTOR", "FACPOB", "FACTOR07"]:
        if fac in df100.columns:
            cols_100.append(fac)
            break
    df100 = df100[cols_100]

    ruta_sum = encontrar_archivo(*carpetas["Sumaria"])
    df_sum = leer_csv_inei(ruta_sum)
    cols_sum = ["CONGLOME", "VIVIENDA", "HOGAR"]
    gasto_encontrada = None
    for g in ["GASHOG2D", "GASHOG1D", "GASHOG2", "GASHOG1"]:
        if g in df_sum.columns:
            gasto_encontrada = g
            break
    if gasto_encontrada:
        df_sum = df_sum.rename(columns={gasto_encontrada: "GASHOG2"})
        cols_sum.append("GASHOG2")
    for m in ["MIEPERHO", "MIEMBRO"]:
        if m in df_sum.columns:
            cols_sum.append(m)
            break
    df_sum = df_sum[cols_sum]

    llave_persona = ["CONGLOME", "VIVIENDA", "HOGAR", "CODPERSO"]
    llave_hogar = ["CONGLOME", "VIVIENDA", "HOGAR"]

    df = df200.merge(df300, on=llave_persona, how="left")
    df = df.merge(df500, on=llave_persona, how="left")
    df = df.merge(df100, on=llave_hogar, how="left")
    df = df.merge(df_sum, on=llave_hogar, how="left")

    df = construir_llave_persona(df)
    df = reparar_variables(df)

    ruta_salida = os.path.join(base_resultados, f"MASTER_{anio}.csv")
    df.to_csv(ruta_salida, index=False, encoding="utf-8-sig")
    print(f"✅ Guardado: {ruta_salida}  ({df.shape[0]:,} x {df.shape[1]})")
    return df

# ============================================================
# 1. CONSTRUIR MASTER 2024 y 2025
# ============================================================

carpetas_2024 = {
    "Mod100": (os.path.join(datos_originales, "966-Modulo01"), "Enaho01-2024-100"),
    "Mod200": (os.path.join(datos_originales, "966-Modulo02"), "Enaho01-2024-200"),
    "Mod300": (os.path.join(datos_originales, "966-Modulo03"), "Enaho01A-2024-300"),
    "Mod500": (os.path.join(datos_originales, "966-Modulo05"), "Enaho01a-2024-500"),
    "Sumaria": (os.path.join(datos_originales, "966-Modulo34"), "Sumaria-2024-12g"),
}
df_master_2024 = construir_master("2024", carpetas_2024)

carpetas_2025 = {
    "Mod100": (os.path.join(datos_originales, "1031-Modulo01-2025"), "Enaho01-2025-100"),
    "Mod200": (os.path.join(datos_originales, "1031-Modulo02-2025"), "Enaho01-2025-200"),
    "Mod300": (os.path.join(datos_originales, "1031-Modulo03-2025"), "Enaho01A-2025-300"),
    "Mod500": (os.path.join(datos_originales, "1031-Modulo05-2025"), "Enaho01a-2025-500"),
    "Sumaria": (os.path.join(datos_originales, "1031-Modulo34-2025"), "Sumaria-2025-12g"),
}
df_master_2025 = construir_master("2025", carpetas_2025)

# ============================================================
# 2. CONSTRUIR PANEL (cruce por llave_persona)
# ============================================================

df24 = pd.read_csv(os.path.join(base_resultados, "MASTER_2024.csv"), encoding="utf-8-sig", low_memory=False)
df25 = pd.read_csv(os.path.join(base_resultados, "MASTER_2025.csv"), encoding="utf-8-sig", low_memory=False)

df24 = construir_llave_persona(df24)
df25 = construir_llave_persona(df25)

match = df24[["llave_persona"]].drop_duplicates().merge(
    df25[["llave_persona"]].drop_duplicates(), on="llave_persona", how="inner"
)

cols_2024 = ["llave_persona", "P203", "P207", "Edad", "Ocupado", "Informal",
             "TenenciaBilletera", "UsoBilletera", "CreditoFormal", "FACTOR07", "P301A",
             "ESTRATO", "DOMINIO", "MIEPERHO", "CONGLOME"]
df_2024_sub = df24[df24["llave_persona"].isin(match["llave_persona"])][cols_2024].rename(
    columns={"CreditoFormal": "CreditoFormal_2024"}
)

cols_2025 = ["llave_persona", "CreditoFormal"]
if "CreditoInformal" in df25.columns:
    cols_2025.append("CreditoInformal")
df_2025_sub = df25[cols_2025].rename(
    columns={"CreditoFormal": "CreditoFormal_2025", "CreditoInformal": "CreditoInformal_2025"}
)

panel_df = df_2024_sub.merge(df_2025_sub, on="llave_persona", how="left")
panel_df.to_csv(os.path.join(base_resultados, "PANEL_2024_2025.csv"), index=False, encoding="utf-8-sig")
print(f"✅ PANEL_2024_2025.csv guardado ({panel_df.shape[0]:,} filas)")

# ============================================================
# 3. FILTRAR MUESTRA ANALÍTICA
# ============================================================

df = pd.read_csv(os.path.join(base_resultados, "PANEL_2024_2025.csv"), encoding="utf-8-sig", low_memory=False)

df["P203_num"] = pd.to_numeric(df["P203"], errors="coerce")
df["Ocupado_num"] = pd.to_numeric(df["Ocupado"], errors="coerce")
df["CreditoFormal_2024_num"] = pd.to_numeric(df["CreditoFormal_2024"], errors="coerce")
df["CreditoFormal_2025_num"] = pd.to_numeric(df["CreditoFormal_2025"], errors="coerce")
df["Informal_num"] = pd.to_numeric(df["Informal"], errors="coerce")

df = df[df["P203_num"] == 1]
df = df[df["Ocupado_num"] == 1]
df = df[df["CreditoFormal_2024_num"] == 0]
df = df.dropna(subset=["Informal_num", "TenenciaBilletera", "UsoBilletera", "P207", "Edad", "P301A"])

df["NuevoCredito"] = (df["CreditoFormal_2025_num"] == 1).astype(int)

# ============================================================
# 4. RENOMBRAR VARIABLES FINALES (NOMBRES DESCRIPTIVOS)
# ============================================================

diccionario_renombres = {
    # Identificación
    "llave_persona": "id_persona",
    
    # Características de la persona
    "P203": "jefe_hogar",
    "P207": "sexo",
    "Edad": "edad",
    "P301A": "nivel_educativo",
    
    # Condición laboral
    "Ocupado": "ocupado",
    "Informal": "trabajador_informal",
    
    # Billeteras digitales
    "TenenciaBilletera": "tiene_billetera",
    "UsoBilletera": "usa_billetera",
    
    # Crédito
    "CreditoFormal_2024": "credito_formal_2024",
    "CreditoFormal_2025": "credito_formal_2025",
    "CreditoInformal_2025": "credito_informal_2025",
    "NuevoCredito": "nuevo_credito_formal",
    
    # Variables de diseño muestral y geográficas
    "FACTOR07": "factor_expansion",
    "ESTRATO": "estrato",
    "DOMINIO": "dominio",
    "MIEPERHO": "miembros_hogar",
    "CONGLOME": "conglomerado",
    
    # Variables numéricas auxiliares (se eliminan porque ya no son necesarias)
    # "P203_num", "Ocupado_num", "CreditoFormal_2024_num", 
    # "CreditoFormal_2025_num", "Informal_num"
}

# Aplicar renombramiento solo a las columnas que existen
columnas_a_renombrar = {k: v for k, v in diccionario_renombres.items() if k in df.columns}
df = df.rename(columns=columnas_a_renombrar)

# Eliminar variables numéricas auxiliares que ya no son necesarias
columnas_auxiliares = ["P203_num", "Ocupado_num", "CreditoFormal_2024_num", 
                       "CreditoFormal_2025_num", "Informal_num"]
columnas_a_eliminar = [col for col in columnas_auxiliares if col in df.columns]
df = df.drop(columns=columnas_a_eliminar)

# ============================================================
# 5. GUARDAR BASE FINAL CON NOMBRES DESCRIPTIVOS
# ============================================================

df.to_csv(os.path.join(base_resultados, "BASE_REGRESIONES.csv"), index=False, encoding="utf-8-sig")
print(f"✅ BASE_REGRESIONES.csv guardado ({df.shape[0]:,} filas)")

# ============================================================
# 6. MOSTRAR RESUMEN DE VARIABLES FINALES
# ============================================================

print("\n" + "="*70)
print("VARIABLES FINALES EN BASE_REGRESIONES.csv")
print("="*70)
for i, col in enumerate(df.columns, 1):
    print(f"{i:>3}. {col}")

print("\n" + "="*70)
print("PROCESO COMPLETADO. ARCHIVOS GENERADOS:")
print("="*70)
print(f"📁 {os.path.join(base_resultados, 'MASTER_2024.csv')}")
print(f"📁 {os.path.join(base_resultados, 'MASTER_2025.csv')}")
print(f"📁 {os.path.join(base_resultados, 'PANEL_2024_2025.csv')}")
print(f"📁 {os.path.join(base_resultados, 'BASE_REGRESIONES.csv')}")

Cargando módulos 2024...
✅ Guardado: C:\Users\Dafne\Documents\GitHub\Base-de-datos-investigacion-Economica\Base24_25\MASTER_2024.csv  (117,721 x 46)
Cargando módulos 2025...
✅ Guardado: C:\Users\Dafne\Documents\GitHub\Base-de-datos-investigacion-Economica\Base24_25\MASTER_2025.csv  (115,145 x 46)
✅ PANEL_2024_2025.csv guardado (32,079 filas)
✅ BASE_REGRESIONES.csv guardado (6,361 filas)

VARIABLES FINALES EN BASE_REGRESIONES.csv
  1. id_persona
  2. jefe_hogar
  3. sexo
  4. edad
  5. ocupado
  6. trabajador_informal
  7. tiene_billetera
  8. usa_billetera
  9. credito_formal_2024
 10. factor_expansion
 11. nivel_educativo
 12. estrato
 13. dominio
 14. miembros_hogar
 15. conglomerado
 16. credito_formal_2025
 17. credito_informal_2025
 18. nuevo_credito_formal

PROCESO COMPLETADO. ARCHIVOS GENERADOS:
📁 C:\Users\Dafne\Documents\GitHub\Base-de-datos-investigacion-Economica\Base24_25\MASTER_2024.csv
📁 C:\Users\Dafne\Documents\GitHub\Base-de-datos-investigacion-Economica\Base24_25\MASTER

In [8]:
import pandas as pd

ruta = r"C:\Users\Dafne\Documents\GitHub\Base-de-datos-investigacion-Economica\Base24_25\BASE_REGRESIONES.csv"
df = pd.read_csv(ruta, encoding="utf-8-sig")

# Guardar toda la información en un archivo de texto
with open(r"C:\Users\Dafne\Documents\GitHub\Base-de-datos-investigacion-Economica\Base24_25\diccionario_variables.txt", "w", encoding="utf-8") as f:
    f.write("="*60 + "\n")
    f.write("VARIABLES EN BASE_REGRESIONES.csv\n")
    f.write("="*60 + "\n")
    for i, col in enumerate(df.columns, 1):
        f.write(f"{i:>3}. {col}\n")
    f.write(f"\nTotal: {len(df.columns)} variables\n\n")
    
    f.write("="*60 + "\n")
    f.write("PRIMERAS 5 FILAS\n")
    f.write("="*60 + "\n")
    f.write(df.head().to_string() + "\n\n")
    
    f.write("="*60 + "\n")
    f.write("ESTADÍSTICAS DESCRIPTIVAS\n")
    f.write("="*60 + "\n")
    f.write(df.describe().to_string() + "\n\n")
    
    f.write("="*60 + "\n")
    f.write("VALORES ÚNICOS DE VARIABLES CLAVE\n")
    f.write("="*60 + "\n")
    variables_clave = ["Billetera", "Informal", "Ocupado", "NuevoCredito", "CreditoFormal_2024", "CreditoFormal_2025"]
    for var in variables_clave:
        if var in df.columns:
            f.write(f"\n{var}:\n")
            f.write(df[var].value_counts(dropna=False).to_string() + "\n")

print("✅ Diccionario guardado en: Base24_25/diccionario_variables.txt")

✅ Diccionario guardado en: Base24_25/diccionario_variables.txt


In [9]:
import pandas as pd
import numpy as np

# Leer el archivo
ruta = r"C:\Users\Dafne\Documents\GitHub\Base-de-datos-investigacion-Economica\Base24_25\BASE_REGRESIONES.csv"
df = pd.read_csv(ruta, encoding="utf-8-sig")

# ============================================================
# 1. REVISAR VALORES FALTANTES (MISSING) POR VARIABLE
# ============================================================
print("="*70)
print("VALORES FALTANTES (MISSING) POR VARIABLE")
print("="*70)

missing = df.isna().sum()
missing_pct = (missing / len(df)) * 100

missing_df = pd.DataFrame({
    'Faltantes': missing,
    'Porcentaje': missing_pct
})
missing_df = missing_df[missing_df['Faltantes'] > 0].sort_values('Faltantes', ascending=False)

if len(missing_df) == 0:
    print("✅ ¡No hay valores faltantes en ninguna variable!")
else:
    print(missing_df)

# ============================================================
# 2. VERIFICACIÓN DE FILTROS APLICADOS
# ============================================================
print("\n" + "="*70)
print("VERIFICACIÓN DE FILTROS APLICADOS")
print("="*70)

print(f"Jefes de hogar: {df['jefe_hogar'].sum():,} de {len(df):,} ({df['jefe_hogar'].sum()/len(df)*100:.1f}%)")
print(f"Ocupados: {df['ocupado'].sum():,} de {len(df):,} ({df['ocupado'].sum()/len(df)*100:.1f}%)")
print(f"Sin crédito formal en 2024: {(df['credito_formal_2024'] == 0).sum():,} de {len(df):,} ({(df['credito_formal_2024'] == 0).sum()/len(df)*100:.1f}%)")

# Variables clave sin missing
vars_clave = ['trabajador_informal', 'tiene_billetera', 'usa_billetera', 'sexo', 'edad', 'nivel_educativo']
sin_missing = df.dropna(subset=vars_clave).shape[0]
print(f"Sin missing en variables clave: {sin_missing:,} de {len(df):,} ({sin_missing/len(df)*100:.1f}%)")

# ============================================================
# 3. ANÁLISIS DE VARIABLES DE BILLETERA
# ============================================================
print("\n" + "="*70)
print("ANÁLISIS DE VARIABLES DE BILLETERA")
print("="*70)

print("Variables de billetera encontradas:")
if 'tiene_billetera' in df.columns:
    print(f"  - tiene_billetera (tenencia): {df['tiene_billetera'].sum():,} usuarios ({df['tiene_billetera'].mean():.1%})")
if 'usa_billetera' in df.columns:
    print(f"  - usa_billetera (uso efectivo): {df['usa_billetera'].sum():,} usuarios ({df['usa_billetera'].mean():.1%})")

# Verificar si son iguales
if 'tiene_billetera' in df.columns and 'usa_billetera' in df.columns:
    print("\nComparación entre tenencia y uso:")
    print(pd.crosstab(df['tiene_billetera'], df['usa_billetera'], margins=True))

# ============================================================
# 4. ESTADÍSTICAS DE VARIABLES CLAVE
# ============================================================
print("\n" + "="*70)
print("ESTADÍSTICAS DE VARIABLES CLAVE")
print("="*70)

# Variable dependiente
if 'nuevo_credito_formal' in df.columns:
    print(f"\n📌 nuevo_credito_formal (entrada al crédito formal):")
    print(f"  - Total: {df['nuevo_credito_formal'].sum():,} personas")
    print(f"  - Tasa: {df['nuevo_credito_formal'].mean():.2%}")

# Informalidad
if 'trabajador_informal' in df.columns:
    print(f"\n📌 trabajador_informal (informalidad):")
    print(f"  - Informales: {df['trabajador_informal'].sum():,} personas")
    print(f"  - Tasa: {df['trabajador_informal'].mean():.2%}")

# Edad
if 'edad' in df.columns:
    print(f"\n📌 edad:")
    print(f"  - Media: {df['edad'].mean():.1f} años")
    print(f"  - Mediana: {df['edad'].median():.1f} años")
    print(f"  - Mínimo: {df['edad'].min():.0f} años")
    print(f"  - Máximo: {df['edad'].max():.0f} años")

# ============================================================
# 5. TABLA CRUZADA: BILLETERA × INFORMALIDAD
# ============================================================
print("\n" + "="*70)
print("TABLA CRUZADA: BILLETERA × INFORMALIDAD")
print("="*70)

if 'tiene_billetera' in df.columns and 'trabajador_informal' in df.columns:
    print("\n1. Tenencia de billetera vs Informalidad:")
    print(pd.crosstab(df['tiene_billetera'], df['trabajador_informal'], margins=True, normalize='columns'))

if 'usa_billetera' in df.columns and 'trabajador_informal' in df.columns:
    print("\n2. Uso de billetera vs Informalidad:")
    print(pd.crosstab(df['usa_billetera'], df['trabajador_informal'], margins=True, normalize='columns'))

# ============================================================
# 6. PROBABILIDADES CONDICIONALES (corazón de la investigación)
# ============================================================
print("\n" + "="*70)
print("PROBABILIDADES CONDICIONALES")
print("="*70)

# Usando usa_billetera (recomendada)
if 'usa_billetera' in df.columns and 'trabajador_informal' in df.columns:
    print("\n📊 Usando 'usa_billetera' (uso efectivo):")
    
    # Informales con billetera
    mask = (df['trabajador_informal'] == 1) & (df['usa_billetera'] == 1)
    p1 = df.loc[mask, 'nuevo_credito_formal'].mean()
    n1 = mask.sum()
    print(f"  Informales, usa_billetera=1: {p1:.2%} (N={n1:,})")
    
    # Informales sin billetera
    mask = (df['trabajador_informal'] == 1) & (df['usa_billetera'] == 0)
    p2 = df.loc[mask, 'nuevo_credito_formal'].mean()
    n2 = mask.sum()
    print(f"  Informales, usa_billetera=0: {p2:.2%} (N={n2:,})")
    print(f"  → Diferencia bruta (informales): {(p1-p2)*100:+.2f} p.p.")
    
    # Formales con billetera
    mask = (df['trabajador_informal'] == 0) & (df['usa_billetera'] == 1)
    p3 = df.loc[mask, 'nuevo_credito_formal'].mean()
    n3 = mask.sum()
    print(f"\n  Formales, usa_billetera=1: {p3:.2%} (N={n3:,})")
    
    # Formales sin billetera
    mask = (df['trabajador_informal'] == 0) & (df['usa_billetera'] == 0)
    p4 = df.loc[mask, 'nuevo_credito_formal'].mean()
    n4 = mask.sum()
    print(f"  Formales, usa_billetera=0: {p4:.2%} (N={n4:,})")
    print(f"  → Diferencia bruta (formales): {(p3-p4)*100:+.2f} p.p.")

# ============================================================
# 7. RESUMEN FINAL
# ============================================================
print("\n" + "="*70)
print("RESUMEN FINAL")
print("="*70)

print(f"📊 Total de observaciones: {len(df):,}")
print(f"📊 Total de variables: {len(df.columns)}")
print(f"📊 Filas sin missing: {df.dropna().shape[0]:,} ({(df.dropna().shape[0]/len(df))*100:.1f}%)")
print(f"📊 Missing totales: {missing.sum():,} ({(missing.sum() / (len(df) * len(df.columns))) * 100:.2f}%)")

# ============================================================
# 8. RECOMENDACIÓN FINAL
# ============================================================
print("\n" + "="*70)
print("RECOMENDACIÓN PARA EL ANÁLISIS")
print("="*70)

if 'usa_billetera' in df.columns:
    print("✅ Usar 'usa_billetera' como variable principal de billetera digital")
    print("   (mide el USO EFECTIVO, no solo tenencia)")
    print("\n   Variables clave para el modelo:")
    print("   - Dependiente: nuevo_credito_formal")
    print("   - Independiente principal: usa_billetera")
    print("   - Moderador: trabajador_informal")
    print("   - Controles: edad, sexo, nivel_educativo, estrato, dominio, miembros_hogar")
    print("   - Cluster: conglomerado")
    print("   - Ponderación: factor_expansion")
elif 'tiene_billetera' in df.columns:
    print("⚠️ Usar 'tiene_billetera' (es la única disponible)")
    print("   (mide TENENCIA, no uso efectivo)")

VALORES FALTANTES (MISSING) POR VARIABLE
✅ ¡No hay valores faltantes en ninguna variable!

VERIFICACIÓN DE FILTROS APLICADOS
Jefes de hogar: 6,361 de 6,361 (100.0%)
Ocupados: 6,361 de 6,361 (100.0%)
Sin crédito formal en 2024: 6,361 de 6,361 (100.0%)
Sin missing en variables clave: 6,361 de 6,361 (100.0%)

ANÁLISIS DE VARIABLES DE BILLETERA
Variables de billetera encontradas:
  - tiene_billetera (tenencia): 1,495 usuarios (23.5%)
  - usa_billetera (uso efectivo): 872 usuarios (13.7%)

Comparación entre tenencia y uso:
usa_billetera       0    1   All
tiene_billetera                 
0                4859    7  4866
1                 630  865  1495
All              5489  872  6361

ESTADÍSTICAS DE VARIABLES CLAVE

📌 nuevo_credito_formal (entrada al crédito formal):
  - Total: 588 personas
  - Tasa: 9.24%

📌 trabajador_informal (informalidad):
  - Informales: 5,029.0 personas
  - Tasa: 79.06%

📌 edad:
  - Media: 52.8 años
  - Mediana: 52.0 años
  - Mínimo: 18 años
  - Máximo: 94 años

TA